## Topic Modeling with LDA and Introduction to STM

This Jupyter Notebook explores topic modeling using **Latent Dirichlet Allocation (LDA)**.
It also briefly introduces **Structural Topic Modeling (STM)** and provides visualizations to interpret results.

We will cover:
1. Data preprocessing
2. LDA model training
3. Visualizing LDA topics
4. Brief introduction to STM


In [ ]:
# Install required packages (uncomment if needed)
#!pip install pandas numpy gensim pyLDAvis matplotlib seaborn bertopic

In [2]:
pip install pyLDAvis

  Using cached pyLDAvis-3.4.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached funcy-2.0-py2.py3-none-any.whl.metadata (5.9 kB)
Using cached pyLDAvis-3.4.1-py3-none-any.whl (2.6 MB)
Using cached funcy-2.0-py2.py3-none-any.whl (30 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
import gensim
import gensim.corpora as corpora
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import LdaModel
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from bertopic import BERTopic
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/deniseroth/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/deniseroth/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### **1. Load Dataset**

Ensure the dataset has a 'text' column containing textual data for topic modeling.


In [ ]:
df = pd.read_csv("/Users/deniseroth/Documents/Downloads/mental_health_sentiment.csv")
print("Dataset Loaded Successfully!")
print(df.head())

Dataset Loaded Successfully!
  id                                               text sentiment  \
0  0                                         oh my gosh   Anxiety   
1  1  trouble sleeping, confused mind, restless hear...   Anxiety   
2  2  All wrong, back off dear, forward doubt. Stay ...   Anxiety   
3  3  I've shifted my focus to something else but I'...   Anxiety   
4  4  I'm restless and restless, it's been a month n...   Anxiety   

   condition_binary  
0                 1  
1                 1  
2                 1  
3                 1  
4                 1  


### **2. Preprocess Text Data**

Text preprocessing includes:
- Tokenization
- Lowercasing
- Stopword removal
- Creating a dictionary and corpus

In [3]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    tokens = word_tokenize(str(text).lower())  # Lowercase and tokenize
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]  # Remove stopwords and non-alphabetic tokens
    return tokens

# Apply preprocessing
df['processed_text'] = df['text'].apply(preprocess_text)

# Create dictionary and corpus
dictionary = corpora.Dictionary(df['processed_text'])
corpus = [dictionary.doc2bow(text) for text in df['processed_text']]

### **3. Train LDA Model**

LDA (Latent Dirichlet Allocation) is a probabilistic model that discovers latent topics in text.


In [4]:
num_topics = 5  # Define the number of topics
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, passes=10, random_state=42)

# Print topics
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}: {topic}")

Topic 0: 0.015*"help" + 0.013*"depression" + 0.011*"bipolar" + 0.009*"meds" + 0.008*"everyone" + 0.008*"anyone" + 0.008*"therapy" + 0.007*"stress" + 0.007*"mental" + 0.007*"need"
Topic 1: 0.037*"like" + 0.031*"feel" + 0.018*"know" + 0.017*"want" + 0.016*"life" + 0.015*"people" + 0.015*"even" + 0.011*"really" + 0.011*"get" + 0.010*"think"
Topic 2: 0.054*"anxiety" + 0.015*"ago" + 0.013*"feeling" + 0.013*"heart" + 0.012*"panic" + 0.011*"symptoms" + 0.010*"anxious" + 0.008*"social" + 0.008*"doctor" + 0.008*"anyone"
Topic 3: 0.039*"im" + 0.022*"amp" + 0.021*"na" + 0.018*"put" + 0.016*"wish" + 0.016*"dont" + 0.012*"gon" + 0.011*"cant" + 0.009*"experiences" + 0.009*"wan"
Topic 4: 0.014*"time" + 0.012*"get" + 0.012*"work" + 0.012*"back" + 0.011*"got" + 0.010*"go" + 0.010*"day" + 0.010*"take" + 0.009*"going" + 0.008*"last"


### **4. Visualize LDA Topics**

Use pyLDAvis to visualize topics and their distribution.


In [5]:
lda_display = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.display(lda_display)

/opt/anaconda3/envs/data_types_for_health_promotion/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarning:

This process (pid=48500) is multi-threaded, use of fork() may lead to deadlocks in the child.

/opt/anaconda3/envs/data_types_for_health_promotion/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarning:

This process (pid=48500) is multi-threaded, use of fork() may lead to deadlocks in the child.

/opt/anaconda3/envs/data_types_for_health_promotion/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarning:

This process (pid=48500) is multi-threaded, use of fork() may lead to deadlocks in the child.

/opt/anaconda3/envs/data_types_for_health_promotion/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarning:

This process (pid=48500) is multi-threaded, use of fork() may lead to deadlocks in the child.

/opt/anaconda3/envs/data

### **5. Brief Introduction to Structural Topic Modeling (STM) with BERTopic**

STM extends LDA by incorporating metadata (e.g., time, author) into the topic modeling process.
It is useful for analyzing how topics change over time or across groups.

STM is implemented in R (package: stm), but there are limited Python implementations.
To use STM in Python, you may need to use **bertopic** or adapt LDA to include document-level metadata.

For more on STM: https://www.structuraltopicmodel.com

BERTopic extends topic modeling by incorporating contextual embeddings and transformers,
making it useful for Structural Topic Modeling (STM).
It allows analysis of how topics vary with metadata (e.g., time, authors, sentiment).


In [ ]:
df['text'] = df['text'].astype(str).fillna("")
# Instantiate BERTopic model
bertopic_model = BERTopic()

# Fit BERTopic model
bertopic_topics, bertopic_probs = bertopic_model.fit_transform(df['text'])

# Visualizing BERTopic (STM) results
# bertopic_model.visualize_barchart()
# bertopic_model.visualize_topics()
# bertopic_model.visualize_distribution(bertopic_probs)

### **6. Using BERTopic for Structural Topic Modeling (STM)**

To analyze how topics evolve over metadata (e.g., time, categories), we can leverage BERTopic’s dynamic topic modeling.
This allows us to track topic distribution across different segments of the data.

Example: If the dataset contains a 'date' or 'category' column, we can examine topic evolution.


In [12]:
# Example: Analyzing topic evolution over time (if time data is available)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])  # Ensure date column is in datetime format
    topics_over_time = bertopic_model.topics_over_time(df['text'], df['date'])
    bertopic_model.visualize_topics_over_time(topics_over_time)

# Example: Analyzing topic distribution across categories (if available)
if 'category' in df.columns:
    topics_per_class = bertopic_model.topics_per_class(df['text'], classes=df['category'])
    bertopic_model.visualize_barchart(topics_per_class=topics_per_class)

print("LDA and BERTopic-based STM modeling complete!")

LDA and BERTopic-based STM modeling complete!
